In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from src.common.utils import get_root_directory
from src.visualisation.plotter import Plotter
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import calendar
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy
import gensim
from gensim.utils import simple_preprocess
import numpy as np

## Data Loading

In [17]:
root_dir = get_root_directory()

## ETH

In [46]:
eth_etherscan = pd.read_csv(f"{root_dir}\data\processed\ETH_data\ETH_etherscan.csv")
eth_ol = pd.read_csv(f"{root_dir}\data\processed\ETH_data\ETH_oklink.csv")
eth_etherscan.replace(np.nan, 0, inplace=True)
eth_etherscan.replace(0, np.nan, inplace=True)
eth_ol.replace(np.nan, 0, inplace=True)
eth_ol.replace(0, np.nan, inplace=True)

In [47]:
eth_etherscan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        2467 non-null   object 
 1   Open                        2467 non-null   float64
 2   High                        2467 non-null   float64
 3   Low                         2467 non-null   float64
 4   Close                       2467 non-null   float64
 5   Adj Close                   2467 non-null   float64
 6   Volume                      2467 non-null   int64  
 7   AddressCount                2467 non-null   int64  
 8   AverageDailyTransactionFee  2467 non-null   float64
 9   AvgGasPrice                 2467 non-null   int64  
 10  BlockCountRewards           2467 non-null   int64  
 11  BlockDifficulty             1772 non-null   float64
 12  BlockReward                 1772 non-null   float64
 13  BlockSize                   2467 

In [48]:
eth_etherscan.drop(columns=["Adj Close", "AddressCount", "BlockReward", "Ethersupply2", "BlockDifficulty", "NetworkHash", "TxGrowth", "Uncles", "TransactionFee"], inplace=True)

In [49]:
eth_etherscan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        2467 non-null   object 
 1   Open                        2467 non-null   float64
 2   High                        2467 non-null   float64
 3   Low                         2467 non-null   float64
 4   Close                       2467 non-null   float64
 5   Volume                      2467 non-null   int64  
 6   AverageDailyTransactionFee  2467 non-null   float64
 7   AvgGasPrice                 2467 non-null   int64  
 8   BlockCountRewards           2467 non-null   int64  
 9   BlockSize                   2467 non-null   int64  
 10  BlockTime                   2467 non-null   float64
 11  DailyActiveEthAddress       2467 non-null   int64  
 12  deployed contracts          2467 non-null   int64  
 13  GasLimit                    2467 

In [50]:
columns={'Open': 'YF_Op', 'High': 'YF_Hi', 'Low': 'YF_Lo', 'Close':'YF_Cls', 'Volume':'YF_Vol', 'AverageDailyTransactionFee':'ES_AvgTransFee','AvgGasPrice':'ES_AvgGasPrc',
        'BlockCountRewards': 'ES_BlkCnt', 'BlockSize':'ES_BlkSz', 'BlockTime':'ES_BlkTm', 'DailyActiveEthAddress':'ES_ActAdd','deployed contracts':'ES_DepCon','GasLimit':'ES_GasLmt',
        'GasUsed': 'ES_GasUsd', 'verified contracts':"ES_VerCon"}
eth_etherscan.rename(columns=columns, inplace=True)

In [51]:
eth_etherscan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2467 non-null   object 
 1   YF_Op           2467 non-null   float64
 2   YF_Hi           2467 non-null   float64
 3   YF_Lo           2467 non-null   float64
 4   YF_Cls          2467 non-null   float64
 5   YF_Vol          2467 non-null   int64  
 6   ES_AvgTransFee  2467 non-null   float64
 7   ES_AvgGasPrc    2467 non-null   int64  
 8   ES_BlkCnt       2467 non-null   int64  
 9   ES_BlkSz        2467 non-null   int64  
 10  ES_BlkTm        2467 non-null   float64
 11  ES_ActAdd       2467 non-null   int64  
 12  ES_DepCon       2467 non-null   int64  
 13  ES_GasLmt       2467 non-null   int64  
 14  ES_GasUsd       2467 non-null   int64  
 15  ES_VerCon       2467 non-null   int64  
dtypes: float64(6), int64(9), object(1)
memory usage: 308.5+ KB


In [52]:
eth_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 21 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Date                                               2467 non-null   object 
 1   Open                                               2467 non-null   float64
 2   High                                               2467 non-null   float64
 3   Low                                                2467 non-null   float64
 4   Close                                              2467 non-null   float64
 5   Adj Close                                          2467 non-null   float64
 6   Volume                                             2467 non-null   int64  
 7   ETH Gas price                                      2467 non-null   float64
 8   ETH total supply                                   2447 non-null   float64
 9   ETH dail

In [53]:
eth_ol.drop(columns= ['Open',
 'High',
 'Low',
 'Close',
 'Adj Close',
 'Volume',
 'ETH Gas price',
 'ETH total supply',
 'ETH daily transaction fee',
 'Gas utilization rate of ETH',
 'ETH daily new contracts',
 'ETH average daily block size',
 'ETH average computing power of the entire network',
 'ETH average daily transaction fee',
 'ETH mining difficulty',
 'ETH number of daily active addresses'], inplace=True)

In [54]:
eth_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 5 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Date                                           2467 non-null   object 
 1   ETH market cap                                 2467 non-null   float64
 2   Number of new addresses added per day on ETH   2422 non-null   float64
 3   ETH number of daily transactions on the chain  2467 non-null   int64  
 4   ETH daily transaction volume on the chain      2439 non-null   float64
dtypes: float64(3), int64(1), object(1)
memory usage: 96.5+ KB


In [55]:
columns = {'ETH market cap': 'OL_MktCap',"Number of new addresses added per day on ETH":"OL_NewAdd", "ETH number of daily transactions on the chain":"OL_ChnTrans", "ETH daily transaction volume on the chain":"OL_ChnVol"}

In [56]:
eth_ol.rename(columns=columns, inplace=True)

In [57]:
eth_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         2467 non-null   object 
 1   OL_MktCap    2467 non-null   float64
 2   OL_NewAdd    2422 non-null   float64
 3   OL_ChnTrans  2467 non-null   int64  
 4   OL_ChnVol    2439 non-null   float64
dtypes: float64(3), int64(1), object(1)
memory usage: 96.5+ KB


In [58]:
eth = pd.merge(eth_etherscan, eth_ol, on="Date")

In [59]:
eth['Date'] = pd.to_datetime(eth['Date'])
eth = eth.set_index('Date')

In [60]:
eth.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2467 entries, 2017-11-09 to 2024-08-10
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2467 non-null   float64
 1   YF_Hi           2467 non-null   float64
 2   YF_Lo           2467 non-null   float64
 3   YF_Cls          2467 non-null   float64
 4   YF_Vol          2467 non-null   int64  
 5   ES_AvgTransFee  2467 non-null   float64
 6   ES_AvgGasPrc    2467 non-null   int64  
 7   ES_BlkCnt       2467 non-null   int64  
 8   ES_BlkSz        2467 non-null   int64  
 9   ES_BlkTm        2467 non-null   float64
 10  ES_ActAdd       2467 non-null   int64  
 11  ES_DepCon       2467 non-null   int64  
 12  ES_GasLmt       2467 non-null   int64  
 13  ES_GasUsd       2467 non-null   int64  
 14  ES_VerCon       2467 non-null   int64  
 15  OL_MktCap       2467 non-null   float64
 16  OL_NewAdd       2422 non-null   float64
 17  OL_ChnTrans    

In [61]:
eth.interpolate(method="time", inplace =True)

In [62]:
eth.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2467 entries, 2017-11-09 to 2024-08-10
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2467 non-null   float64
 1   YF_Hi           2467 non-null   float64
 2   YF_Lo           2467 non-null   float64
 3   YF_Cls          2467 non-null   float64
 4   YF_Vol          2467 non-null   int64  
 5   ES_AvgTransFee  2467 non-null   float64
 6   ES_AvgGasPrc    2467 non-null   int64  
 7   ES_BlkCnt       2467 non-null   int64  
 8   ES_BlkSz        2467 non-null   int64  
 9   ES_BlkTm        2467 non-null   float64
 10  ES_ActAdd       2467 non-null   int64  
 11  ES_DepCon       2467 non-null   int64  
 12  ES_GasLmt       2467 non-null   int64  
 13  ES_GasUsd       2467 non-null   int64  
 14  ES_VerCon       2467 non-null   int64  
 15  OL_MktCap       2467 non-null   float64
 16  OL_NewAdd       2467 non-null   float64
 17  OL_ChnTrans    

In [117]:
eth['D_AvgPrc'] = eth[['YF_Op', 'YF_Hi', 'YF_Lo','YF_Cls']].mean(axis=1)

In [118]:
eth.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2467 entries, 2017-11-09 to 2024-08-10
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2467 non-null   float64
 1   YF_Hi           2467 non-null   float64
 2   YF_Lo           2467 non-null   float64
 3   YF_Cls          2467 non-null   float64
 4   YF_Vol          2467 non-null   int64  
 5   ES_AvgTransFee  2467 non-null   float64
 6   ES_AvgGasPrc    2467 non-null   int64  
 7   ES_BlkCnt       2467 non-null   int64  
 8   ES_BlkSz        2467 non-null   int64  
 9   ES_BlkTm        2467 non-null   float64
 10  ES_ActAdd       2467 non-null   int64  
 11  ES_DepCon       2467 non-null   int64  
 12  ES_GasLmt       2467 non-null   int64  
 13  ES_GasUsd       2467 non-null   int64  
 14  ES_VerCon       2467 non-null   int64  
 15  OL_MktCap       2467 non-null   float64
 16  OL_NewAdd       2467 non-null   float64
 17  OL_ChnTrans    

In [121]:
eth_filtered = eth.loc[:"2024-04-01"]

In [122]:
eth_filtered.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2336 entries, 2017-11-09 to 2024-04-01
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2336 non-null   float64
 1   YF_Hi           2336 non-null   float64
 2   YF_Lo           2336 non-null   float64
 3   YF_Cls          2336 non-null   float64
 4   YF_Vol          2336 non-null   int64  
 5   ES_AvgTransFee  2336 non-null   float64
 6   ES_AvgGasPrc    2336 non-null   int64  
 7   ES_BlkCnt       2336 non-null   int64  
 8   ES_BlkSz        2336 non-null   int64  
 9   ES_BlkTm        2336 non-null   float64
 10  ES_ActAdd       2336 non-null   int64  
 11  ES_DepCon       2336 non-null   int64  
 12  ES_GasLmt       2336 non-null   int64  
 13  ES_GasUsd       2336 non-null   int64  
 14  ES_VerCon       2336 non-null   int64  
 15  OL_MktCap       2336 non-null   float64
 16  OL_NewAdd       2336 non-null   float64
 17  OL_ChnTrans    

## BTC

In [84]:
btc_bic = pd.read_csv(f"{root_dir}\data\processed\BTC_data\BTC_bitinfocharts.csv")
btc_ol = pd.read_csv(f"{root_dir}\data\processed\BTC_data\BTC_oklink.csv")
btc_bic.replace(np.nan, 0, inplace=True)
btc_bic.replace(0, np.nan, inplace=True)
btc_ol.replace(np.nan, 0, inplace=True)
btc_ol.replace(0, np.nan, inplace=True)

In [85]:
btc_bic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               2467 non-null   object 
 1   Open               2467 non-null   float64
 2   High               2467 non-null   float64
 3   Low                2467 non-null   float64
 4   Close              2467 non-null   float64
 5   Adj Close          2467 non-null   float64
 6   Volume             2467 non-null   int64  
 7   Transactions       2467 non-null   float64
 8   Block Size         2467 non-null   float64
 9   Difficulty         2467 non-null   float64
 10  Hashrate           2467 non-null   float64
 11  Active Addressses  2465 non-null   float64
dtypes: float64(10), int64(1), object(1)
memory usage: 231.4+ KB


In [86]:
btc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 19 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Date                                               2467 non-null   object 
 1   Open                                               2467 non-null   float64
 2   High                                               2467 non-null   float64
 3   Low                                                2467 non-null   float64
 4   Close                                              2467 non-null   float64
 5   Adj Close                                          2467 non-null   float64
 6   Volume                                             2467 non-null   int64  
 7   BTC total supply                                   2467 non-null   float64
 8   BTC market cap                                     2467 non-null   float64
 9   BTC aver

In [87]:
btc_ol.drop(columns=[
 'Adj Close',
 'BTC total supply',
 'BTC total addresses',
 'BTC full node data size'], inplace=True)

In [88]:
btc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 15 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Date                                               2467 non-null   object 
 1   Open                                               2467 non-null   float64
 2   High                                               2467 non-null   float64
 3   Low                                                2467 non-null   float64
 4   Close                                              2467 non-null   float64
 5   Volume                                             2467 non-null   int64  
 6   BTC market cap                                     2467 non-null   float64
 7   BTC average daily block size                       2467 non-null   float64
 8   BTC average computing power of the entire network  2467 non-null   float64
 9   BTC aver

In [89]:
columns={'Open': 'YF_Op', 'High': 'YF_Hi', 'Low': 'YF_Lo', 'Close':'YF_Cls', 'Volume':'YF_Vol', 'BTC average daily transaction fee':'OL_AvgTransFee',
         'BTC mining difficulty': 'OL_MinDif', 'BTC average daily block size':'OL_BlkSz', 'BlockTime':'ES_BlkTm', 'BTC number of daily active addresses':'OL_ActAdd','BTC market cap': 'OL_MktCap',"Number of new addresses added per day on BTC":"OL_NewAdd", "BTC number of daily transactions on the chain":"OL_ChnTrans", 
         "BTC daily transaction volume on the chain":"OL_ChnVol", 'BTC average computing power of the entire network':'OL_AvgComPwr'}
btc_ol.rename(columns=columns, inplace=True)

In [90]:
btc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2467 non-null   object 
 1   YF_Op           2467 non-null   float64
 2   YF_Hi           2467 non-null   float64
 3   YF_Lo           2467 non-null   float64
 4   YF_Cls          2467 non-null   float64
 5   YF_Vol          2467 non-null   int64  
 6   OL_MktCap       2467 non-null   float64
 7   OL_BlkSz        2467 non-null   float64
 8   OL_AvgComPwr    2467 non-null   float64
 9   OL_AvgTransFee  2467 non-null   float64
 10  OL_MinDif       2467 non-null   float64
 11  OL_ActAdd       2466 non-null   float64
 12  OL_NewAdd       2467 non-null   int64  
 13  OL_ChnTrans     2467 non-null   int64  
 14  OL_ChnVol       2467 non-null   float64
dtypes: float64(11), int64(3), object(1)
memory usage: 289.2+ KB


In [91]:
btc_bic.drop(columns=['Open',
 'High',
 'Low',
 'Close',
 'Adj Close',
 'Volume',
 'Transactions',
 'Block Size',
 'Difficulty',
 'Active Addressses'], inplace=True)

In [92]:
btc_bic.rename(columns={"Hashrate":"BIC_HshRt"}, inplace=True)

In [93]:
btc = pd.merge(btc_ol, btc_bic, on="Date")

In [94]:
btc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2467 non-null   object 
 1   YF_Op           2467 non-null   float64
 2   YF_Hi           2467 non-null   float64
 3   YF_Lo           2467 non-null   float64
 4   YF_Cls          2467 non-null   float64
 5   YF_Vol          2467 non-null   int64  
 6   OL_MktCap       2467 non-null   float64
 7   OL_BlkSz        2467 non-null   float64
 8   OL_AvgComPwr    2467 non-null   float64
 9   OL_AvgTransFee  2467 non-null   float64
 10  OL_MinDif       2467 non-null   float64
 11  OL_ActAdd       2466 non-null   float64
 12  OL_NewAdd       2467 non-null   int64  
 13  OL_ChnTrans     2467 non-null   int64  
 14  OL_ChnVol       2467 non-null   float64
 15  BIC_HshRt       2467 non-null   float64
dtypes: float64(12), int64(3), object(1)
memory usage: 308.5+ KB


In [95]:
btc['Date'] = pd.to_datetime(btc['Date'])
btc = btc.set_index('Date')

In [96]:
btc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2467 entries, 2017-11-09 to 2024-08-10
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2467 non-null   float64
 1   YF_Hi           2467 non-null   float64
 2   YF_Lo           2467 non-null   float64
 3   YF_Cls          2467 non-null   float64
 4   YF_Vol          2467 non-null   int64  
 5   OL_MktCap       2467 non-null   float64
 6   OL_BlkSz        2467 non-null   float64
 7   OL_AvgComPwr    2467 non-null   float64
 8   OL_AvgTransFee  2467 non-null   float64
 9   OL_MinDif       2467 non-null   float64
 10  OL_ActAdd       2466 non-null   float64
 11  OL_NewAdd       2467 non-null   int64  
 12  OL_ChnTrans     2467 non-null   int64  
 13  OL_ChnVol       2467 non-null   float64
 14  BIC_HshRt       2467 non-null   float64
dtypes: float64(12), int64(3)
memory usage: 308.4 KB


In [97]:
btc.interpolate(method="time", inplace=True)

In [123]:
btc['D_AvgPrc'] = btc[['YF_Op', 'YF_Hi', 'YF_Lo','YF_Cls']].mean(axis=1)

In [124]:
btc_filtered  = btc.loc[:"2024-04-01"]

In [126]:
btc_filtered.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2336 entries, 2017-11-09 to 2024-04-01
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2336 non-null   float64
 1   YF_Hi           2336 non-null   float64
 2   YF_Lo           2336 non-null   float64
 3   YF_Cls          2336 non-null   float64
 4   YF_Vol          2336 non-null   int64  
 5   OL_MktCap       2336 non-null   float64
 6   OL_BlkSz        2336 non-null   float64
 7   OL_AvgComPwr    2336 non-null   float64
 8   OL_AvgTransFee  2336 non-null   float64
 9   OL_MinDif       2336 non-null   float64
 10  OL_ActAdd       2336 non-null   float64
 11  OL_NewAdd       2336 non-null   int64  
 12  OL_ChnTrans     2336 non-null   int64  
 13  OL_ChnVol       2336 non-null   float64
 14  BIC_HshRt       2336 non-null   float64
 15  D_AvgPrc        2336 non-null   float64
dtypes: float64(13), int64(3)
memory usage: 310.2 KB


## LTC

In [127]:
ltc_bic = pd.read_csv(f"{root_dir}\data\processed\LTC_data\LTC_bitinfocharts.csv")
ltc_ol = pd.read_csv(f"{root_dir}\data\processed\LTC_data\LTC_oklink.csv")
ltc_bic.replace(np.nan, 0, inplace=True)
ltc_bic.replace(0, np.nan, inplace=True)
ltc_ol.replace(np.nan, 0, inplace=True)
ltc_ol.replace(0, np.nan, inplace=True)

In [128]:
ltc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 18 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Date                                               2467 non-null   object 
 1   Open                                               2467 non-null   float64
 2   High                                               2467 non-null   float64
 3   Low                                                2467 non-null   float64
 4   Close                                              2467 non-null   float64
 5   Adj Close                                          2467 non-null   float64
 6   Volume                                             2467 non-null   int64  
 7   LTC total supply                                   2466 non-null   float64
 8   LTC market cap                                     2329 non-null   float64
 9   LTC aver

In [129]:
ltc_ol.drop(columns=[
 'Adj Close',
 'LTC total supply',
 'LTC total addresses',
 'LTC full node data size'], inplace=True)

In [130]:
ltc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 14 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Date                                               2467 non-null   object 
 1   Open                                               2467 non-null   float64
 2   High                                               2467 non-null   float64
 3   Low                                                2467 non-null   float64
 4   Close                                              2467 non-null   float64
 5   Volume                                             2467 non-null   int64  
 6   LTC market cap                                     2329 non-null   float64
 7   LTC average daily block size                       2330 non-null   float64
 8   LTC average computing power of the entire network  2330 non-null   float64
 9   LTC aver

In [131]:
columns={'Open': 'YF_Op', 'High': 'YF_Hi', 'Low': 'YF_Lo', 'Close':'YF_Cls', 'Volume':'YF_Vol', 'LTC average daily transaction fee':'OL_AvgTransFee',
         'LTC mining difficulty': 'OL_MinDif', 'LTC average daily block size':'OL_BlkSz', 'LTC number of daily active addresses':'OL_ActAdd','LTC market cap': 'OL_MktCap',"Number of new addresses added per day on LTC":"OL_NewAdd", "LTC number of daily transactions on the chain":"OL_ChnTrans", 
         "LTC daily transaction volume on the chain":"OL_ChnVol", 'LTC average computing power of the entire network':"OL_AvgComPwr"}
ltc_ol.rename(columns=columns, inplace=True)

In [132]:
ltc_ol.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2467 non-null   object 
 1   YF_Op           2467 non-null   float64
 2   YF_Hi           2467 non-null   float64
 3   YF_Lo           2467 non-null   float64
 4   YF_Cls          2467 non-null   float64
 5   YF_Vol          2467 non-null   int64  
 6   OL_MktCap       2329 non-null   float64
 7   OL_BlkSz        2330 non-null   float64
 8   OL_AvgComPwr    2330 non-null   float64
 9   OL_AvgTransFee  2330 non-null   float64
 10  OL_MinDif       2330 non-null   float64
 11  OL_ActAdd       2466 non-null   float64
 12  OL_NewAdd       2330 non-null   float64
 13  OL_ChnTrans     2330 non-null   float64
dtypes: float64(12), int64(1), object(1)
memory usage: 270.0+ KB


In [133]:
ltc_bic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               2467 non-null   object 
 1   Open               2467 non-null   float64
 2   High               2467 non-null   float64
 3   Low                2467 non-null   float64
 4   Close              2467 non-null   float64
 5   Adj Close          2467 non-null   float64
 6   Volume             2467 non-null   int64  
 7   Transactions       2467 non-null   float64
 8   Block Size         2467 non-null   float64
 9   Difficulty         2467 non-null   float64
 10  Hashrate           2467 non-null   float64
 11  Active Addressses  2422 non-null   float64
dtypes: float64(10), int64(1), object(1)
memory usage: 231.4+ KB


In [134]:
ltc_bic.drop(columns=['Open',
 'High',
 'Low',
 'Close',
 'Adj Close',
 'Volume',
 'Transactions',
 'Block Size',
 'Difficulty',
 'Active Addressses'], inplace=True)


In [135]:
ltc_bic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      2467 non-null   object 
 1   Hashrate  2467 non-null   float64
dtypes: float64(1), object(1)
memory usage: 38.7+ KB


In [136]:
ltc_bic.rename(columns={"Hashrate":"BIC_HshRt"}, inplace=True)

In [137]:
ltc = pd.merge(ltc_ol, ltc_bic, on="Date")

In [138]:
ltc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2467 non-null   object 
 1   YF_Op           2467 non-null   float64
 2   YF_Hi           2467 non-null   float64
 3   YF_Lo           2467 non-null   float64
 4   YF_Cls          2467 non-null   float64
 5   YF_Vol          2467 non-null   int64  
 6   OL_MktCap       2329 non-null   float64
 7   OL_BlkSz        2330 non-null   float64
 8   OL_AvgComPwr    2330 non-null   float64
 9   OL_AvgTransFee  2330 non-null   float64
 10  OL_MinDif       2330 non-null   float64
 11  OL_ActAdd       2466 non-null   float64
 12  OL_NewAdd       2330 non-null   float64
 13  OL_ChnTrans     2330 non-null   float64
 14  BIC_HshRt       2467 non-null   float64
dtypes: float64(13), int64(1), object(1)
memory usage: 289.2+ KB


In [139]:
ltc["Date"] = pd.to_datetime(ltc["Date"])

In [140]:
ltc = ltc.set_index("Date")

In [141]:
ltc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2467 entries, 2017-11-09 to 2024-08-10
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2467 non-null   float64
 1   YF_Hi           2467 non-null   float64
 2   YF_Lo           2467 non-null   float64
 3   YF_Cls          2467 non-null   float64
 4   YF_Vol          2467 non-null   int64  
 5   OL_MktCap       2329 non-null   float64
 6   OL_BlkSz        2330 non-null   float64
 7   OL_AvgComPwr    2330 non-null   float64
 8   OL_AvgTransFee  2330 non-null   float64
 9   OL_MinDif       2330 non-null   float64
 10  OL_ActAdd       2466 non-null   float64
 11  OL_NewAdd       2330 non-null   float64
 12  OL_ChnTrans     2330 non-null   float64
 13  BIC_HshRt       2467 non-null   float64
dtypes: float64(13), int64(1)
memory usage: 289.1 KB


In [142]:
ltc['D_AvgPrc'] = ltc[['YF_Op', 'YF_Hi', 'YF_Lo','YF_Cls']].mean(axis=1)

In [143]:
ltc_filtered = ltc.loc[:"2024-04-01"]

In [144]:
ltc_filtered.interpolate(method="time", inplace=True)

C:\Users\rajdh\AppData\Local\Temp\ipykernel_30188\2251465641.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ltc_filtered.interpolate(method="time", inplace=True)


In [145]:
ltc_filtered.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2336 entries, 2017-11-09 to 2024-04-01
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   YF_Op           2336 non-null   float64
 1   YF_Hi           2336 non-null   float64
 2   YF_Lo           2336 non-null   float64
 3   YF_Cls          2336 non-null   float64
 4   YF_Vol          2336 non-null   int64  
 5   OL_MktCap       2336 non-null   float64
 6   OL_BlkSz        2336 non-null   float64
 7   OL_AvgComPwr    2336 non-null   float64
 8   OL_AvgTransFee  2336 non-null   float64
 9   OL_MinDif       2336 non-null   float64
 10  OL_ActAdd       2336 non-null   float64
 11  OL_NewAdd       2336 non-null   float64
 12  OL_ChnTrans     2336 non-null   float64
 13  BIC_HshRt       2336 non-null   float64
 14  D_AvgPrc        2336 non-null   float64
dtypes: float64(14), int64(1)
memory usage: 292.0 KB
